# Gold — business marts

Builds governed, analytics-ready Gold marts from the trusted Silver tables
(`silver_orders`, `silver_transactions`) around a single, consistent revenue
definition.

## Revenue rules (applied in every mart)
- **Eligible order status:** `shipped` (fulfilment complete). `delivered`,
  `placed`, `returned`, and `cancelled` orders are **excluded** from revenue.
  > Assumption — revenue is recognised at shipment. Confirm whether `delivered`
  > should also count before relying on these numbers downstream.
- **Eligible payment status:** `success`. `pending`, `failed`, and `refunded`
  transactions do not add to gross revenue.
- **Gross eligible revenue** = SUM of `success` transaction amounts on `shipped`
  orders, converted to USD.
- **Net revenue** = gross eligible revenue − refunded amount on the same shipped
  orders (refunds reduce net, never gross).
- **Cancellations:** `cancelled` orders contribute `$0` to revenue and are
  tracked via `cancelled_order_count` / `cancellation_rate`.
- **Partial / repeat payments:** an order may have several `success`
  transactions (instalments). They are **summed per order**; transactions are
  aggregated to **exactly one row per `order_id` BEFORE** the join to orders, so
  the many-to-one relationship never inflates order-level fields or revenue.
- **Currency handling:** all amounts convert to USD via the static FX table
  below. Unmapped currencies yield `NULL` and are excluded from revenue — never
  silently treated as USD.

## FX rates to USD (static assumptions, not market data)
| USD | EUR | GBP | INR | JPY | CAD | AUD | SGD | BDT |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1.0 | 1.08 | 1.27 | 0.012 | 0.0067 | 0.73 | 0.66 | 0.74 | 0.0091 |

## Gold marts — grain, unique key, metrics
| Mart | Grain (one row per) | Unique key | Key metrics |
| --- | --- | --- | --- |
| `gold_daily_sales` | `order_date` | `order_date` | order / eligible / cancelled counts, cancellation_rate, gross & net eligible revenue |
| `gold_revenue_breakdown` | `category` × `country` | `(category, country)` | order / customer counts, gross & net revenue, revenue_share |
| `gold_payment_success` | `payment_method` | `payment_method` | transaction / success counts, success_rate, total amount USD |
| `gold_top_customers` | `customer_id` (top 100) | `customer_id` | gross & net eligible spend, eligible order count, spend_rank |

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catalog")
dbutils.widgets.text("schema", "ecom_poc", "Schema")

CATALOG = dbutils.widgets.get("catalog").strip()
SCHEMA = dbutils.widgets.get("schema").strip()

# Static FX rates to USD (assumptions, not market data).
# ELSE NULL: any currency not listed maps to NULL, so SUM() drops it from
# revenue — an unmapped currency can never be silently treated as USD.
FX = ("CASE currency "
      "WHEN 'USD' THEN 1.0 "
      "WHEN 'EUR' THEN 1.08 "
      "WHEN 'GBP' THEN 1.27 "
      "WHEN 'INR' THEN 0.012 "
      "WHEN 'JPY' THEN 0.0067 "
      "WHEN 'CAD' THEN 0.73 "
      "WHEN 'AUD' THEN 0.66 "
      "WHEN 'SGD' THEN 0.74 "
      "WHEN 'BDT' THEN 0.0091 "
      "ELSE NULL END")

In [0]:
# Order-level revenue.
# Transactions are aggregated to ONE row per order_id BEFORE the join, so the
# many-to-one (transactions -> orders) relationship cannot inflate revenue or
# duplicate order-level fields. amount_usd is NULL for unmapped currencies,
# which SUM() ignores (excluded from revenue, never silently USD).

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW txn_converted AS
SELECT order_id, payment_method, txn_status,
       ROUND(amount * {FX}, 2) AS amount_usd
FROM `{CATALOG}`.`{SCHEMA}`.`silver_transactions`
""")

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW order_revenue AS
SELECT
    o.order_id, o.customer_id, o.order_date, o.category, o.country, o.order_status,
    o.quantity, o.unit_price,
    (o.order_status = 'shipped')    AS is_eligible,
    (o.order_status = 'cancelled')  AS is_cancelled,
    COALESCE(p.paid_amount, 0)      AS paid_amount,
    COALESCE(p.refunded_amount, 0)  AS refunded_amount,
    CASE WHEN o.order_status = 'shipped'
         THEN COALESCE(p.paid_amount, 0) ELSE 0 END AS eligible_revenue,
    CASE WHEN o.order_status = 'shipped'
         THEN COALESCE(p.paid_amount, 0) - COALESCE(p.refunded_amount, 0)
         ELSE 0 END AS net_revenue
FROM `{CATALOG}`.`{SCHEMA}`.`silver_orders` o
LEFT JOIN (
    SELECT order_id,
        SUM(CASE WHEN txn_status = 'success'  THEN amount_usd ELSE 0 END) AS paid_amount,
        SUM(CASE WHEN txn_status = 'refunded' THEN amount_usd ELSE 0 END) AS refunded_amount
    FROM txn_converted GROUP BY order_id
) p ON o.order_id = p.order_id
""")

print(f"order_revenue: {spark.sql('SELECT COUNT(*) FROM order_revenue').collect()[0][0]:,} rows")

In [0]:
# 1. gold_daily_sales — daily revenue and cancellation rate
spark.sql(f"""
CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.`gold_daily_sales` USING DELTA AS
SELECT
    order_date,
    CAST(COUNT(*) AS BIGINT) AS order_count,
    CAST(SUM(CASE WHEN is_eligible THEN 1 ELSE 0 END) AS BIGINT) AS eligible_order_count,
    CAST(SUM(CASE WHEN is_cancelled THEN 1 ELSE 0 END) AS BIGINT) AS cancelled_order_count,
    CAST(SUM(CASE WHEN is_cancelled THEN 1 ELSE 0 END) / COUNT(*) AS DECIMAL(9,6)) AS cancellation_rate,
    CAST(SUM(eligible_revenue) AS DECIMAL(28,2)) AS eligible_revenue,
    CAST(SUM(net_revenue) AS DECIMAL(28,2)) AS net_revenue
FROM order_revenue GROUP BY order_date
""")
spark.sql(f"COMMENT ON TABLE `{CATALOG}`.`{SCHEMA}`.`gold_daily_sales` IS 'Daily sales mart. Grain: one row per order_date (unique key). Revenue = success payments on shipped orders, in USD; delivered/placed/returned/cancelled orders are excluded from revenue but counted in order_count.'")
for col, cmt in [
    ("order_date", "Calendar date of the order (DATE). Unique key."),
    ("order_count", "Total orders that day across all statuses."),
    ("eligible_order_count", "Orders with status shipped (revenue-eligible)."),
    ("cancelled_order_count", "Orders with status cancelled."),
    ("cancellation_rate", "cancelled_order_count / order_count, range 0..1."),
    ("eligible_revenue", "Gross USD revenue: sum of success payments on shipped orders, before refunds."),
    ("net_revenue", "Net USD revenue: eligible_revenue minus refunded amounts on shipped orders."),
]:
    spark.sql(f"COMMENT ON COLUMN `{CATALOG}`.`{SCHEMA}`.`gold_daily_sales`.`{col}` IS '{cmt}'")

# 2. gold_revenue_breakdown — revenue by category x country
spark.sql(f"""
CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.`gold_revenue_breakdown` USING DELTA AS
WITH grp AS (
    SELECT category, country,
        CAST(COUNT(*) AS BIGINT) AS order_count,
        CAST(COUNT(DISTINCT customer_id) AS BIGINT) AS customer_count,
        CAST(SUM(eligible_revenue) AS DECIMAL(28,2)) AS eligible_revenue,
        CAST(SUM(net_revenue) AS DECIMAL(28,2)) AS net_revenue
    FROM order_revenue GROUP BY category, country
)
SELECT category, country, order_count, customer_count, eligible_revenue, net_revenue,
    CAST(eligible_revenue / NULLIF(SUM(eligible_revenue) OVER (), 0) AS DECIMAL(9,6)) AS revenue_share
FROM grp
""")
spark.sql(f"COMMENT ON TABLE `{CATALOG}`.`{SCHEMA}`.`gold_revenue_breakdown` IS 'Revenue by product category and country. Grain: one row per (category, country) unique key. Revenue = success payments on shipped orders, USD.'")
for col, cmt in [
    ("category", "Product category from silver_orders. Part of unique key."),
    ("country", "Order country (uppercased) from silver_orders. Part of unique key."),
    ("order_count", "Total orders for this category x country (all statuses)."),
    ("customer_count", "Distinct customers with an order in this group."),
    ("eligible_revenue", "Gross USD revenue: success payments on shipped orders, before refunds."),
    ("net_revenue", "Net USD revenue: eligible_revenue minus refunds on shipped orders."),
    ("revenue_share", "eligible_revenue / total eligible_revenue across all groups; sums to 1.0."),
]:
    spark.sql(f"COMMENT ON COLUMN `{CATALOG}`.`{SCHEMA}`.`gold_revenue_breakdown`.`{col}` IS '{cmt}'")

# 3. gold_payment_success — payment success rate by method
# Spans ALL transactions (not just shipped orders): measures payment-method
# health across the full population. total_amount_usd excludes any unmapped
# currency (NULL amount_usd); transaction_count includes all rows.
spark.sql(f"""
CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.`gold_payment_success` USING DELTA AS
SELECT payment_method,
    CAST(COUNT(*) AS BIGINT) AS transaction_count,
    CAST(SUM(CASE WHEN txn_status = 'success' THEN 1 ELSE 0 END) AS BIGINT) AS successful_count,
    CAST(SUM(CASE WHEN txn_status = 'success' THEN 1 ELSE 0 END) / COUNT(*) AS DECIMAL(9,6)) AS success_rate,
    CAST(SUM(amount_usd) AS DECIMAL(28,2)) AS total_amount_usd
FROM txn_converted GROUP BY payment_method
""")
spark.sql(f"COMMENT ON TABLE `{CATALOG}`.`{SCHEMA}`.`gold_payment_success` IS 'Payment success rate by method. Grain: one row per payment_method (unique key). Covers ALL transactions, not only shipped orders.'")
for col, cmt in [
    ("payment_method", "Payment method from silver_transactions. Unique key."),
    ("transaction_count", "All transactions for this method (every status and currency)."),
    ("successful_count", "Transactions with txn_status = success."),
    ("success_rate", "successful_count / transaction_count, range 0..1."),
    ("total_amount_usd", "Sum of USD-converted amounts; excludes unmapped-currency rows (NULL)."),
]:
    spark.sql(f"COMMENT ON COLUMN `{CATALOG}`.`{SCHEMA}`.`gold_payment_success`.`{col}` IS '{cmt}'")

# 4. gold_top_customers — top 100 by eligible spend
spark.sql(f"""
CREATE OR REPLACE TABLE `{CATALOG}`.`{SCHEMA}`.`gold_top_customers` USING DELTA AS
SELECT * FROM (
    SELECT customer_id,
        CAST(SUM(eligible_revenue) AS DECIMAL(28,2)) AS eligible_spend,
        CAST(SUM(net_revenue) AS DECIMAL(28,2)) AS net_spend,
        CAST(SUM(CASE WHEN is_eligible THEN 1 ELSE 0 END) AS BIGINT) AS eligible_order_count,
        RANK() OVER (ORDER BY SUM(eligible_revenue) DESC) AS spend_rank
    FROM order_revenue WHERE customer_id IS NOT NULL GROUP BY customer_id
    HAVING SUM(eligible_revenue) > 0
) WHERE spend_rank <= 100
""")
spark.sql(f"COMMENT ON TABLE `{CATALOG}`.`{SCHEMA}`.`gold_top_customers` IS 'Top 100 customers by eligible spend. Grain: one row per customer_id (unique key), ranked by gross eligible spend DESC. Only customers with positive eligible revenue.'")
for col, cmt in [
    ("customer_id", "Customer identifier from silver_orders. Unique key."),
    ("eligible_spend", "Gross USD eligible revenue for this customer (success payments on shipped orders, pre-refund)."),
    ("net_spend", "Net USD spend: eligible_spend minus refunds on shipped orders."),
    ("eligible_order_count", "Number of shipped orders for this customer."),
    ("spend_rank", "RANK() by eligible_spend DESC; 1 = top spender. Ties can yield more than 100 rows at the boundary."),
]:
    spark.sql(f"COMMENT ON COLUMN `{CATALOG}`.`{SCHEMA}`.`gold_top_customers`.`{col}` IS '{cmt}'")

print("All 4 gold marts written")

In [0]:
# Row counts + sample outputs to verify each mart
for t in ["gold_daily_sales", "gold_revenue_breakdown", "gold_payment_success", "gold_top_customers"]:
    tbl = spark.table(f"`{CATALOG}`.`{SCHEMA}`.`{t}`")
    print(f"{t}: {tbl.count():,} rows")
    display(tbl.limit(5))

## Governed Reporting Layer — AI/BI Dashboard & Genie Agent

This section documents the native reporting layer built on top of the Gold marts.

### AI/BI Dashboard
**Dashboard:** [E-commerce Sales Dashboard](#dashboard-3215078311518684) (UUID: `01f1e319778f152eaa1366eadfa17014`)

The dashboard was enhanced to meet governed Gold-layer reporting requirements:

**5 KPI counters** (exceeds minimum 4):
1. **Total Eligible Revenue** — `SUM(eligible_revenue)` = $6,528,821.67
2. **Net Revenue** — `SUM(net_revenue)` = $6,191,944.52 (added per Gold business definition)
3. **Total Orders** — `SUM(order_count)` = 952,374
4. **Eligible Orders** — `SUM(eligible_order_count)` = 171,520
5. **Overall Cancellation Rate** — `SUM(cancelled_order_count)/SUM(order_count)` = 12.0%
   (Fixed from `AVG(cancellation_rate)` — the Gold business definition requires a
   weighted ratio, not an unweighted average of daily rates)

**2 time-series trend visuals:**
1. Revenue Trend (Monthly) — `DATE_TRUNC('MONTH', order_date)` vs `SUM(eligible_revenue)`
2. Cancellation Rate Trend — monthly `SUM(cancelled)/SUM(total)` (fixed from AVG)

**3 breakdown visuals:**
1. Revenue by Category (bar) — `gold_revenue_breakdown`
2. Payment Success Rate (bar) — `gold_payment_success` (fixed from `SUM(success_rate)`
   to use the pre-calculated `success_rate` field directly)
3. Revenue by Category × Country (table)

**1 shared filter:** Date-range filter on `order_date` (2023-01-01 to 2026-10-31),
applied to all widgets sourced from `gold_daily_sales`.

**Metric fixes applied for Gold business definition compliance:**
- Cancellation rate: `SUM(cancelled)/SUM(total)` (weighted) replaces `AVG(rate)` (unweighted)
- Payment success rate: direct `success_rate` field replaces `SUM(success_rate)`
- Net revenue counter added (was missing from original dashboard)

### Genie Agent
**Genie Agent:** E-commerce Gold Sales Analytics (ID: `01f1e34620fa133f9a73a768020d9ad3`)

Configured over all 4 Gold tables. Three business questions asked and cross-checked
against trusted SQL (see cells below):
1. Total eligible revenue → $6,528,821.67 ✓
2. Top category by eligible revenue → Sports ($944,994.99) ✓
3. Card payment success rate → 77.99% (302,339 / 387,641) ✓

### Power BI cross-check
No `.pbix` file was found in the Databricks workspace. The reconciliation notebook
produces the verified Databricks SQL baseline (identical filters, same FX rates)
for manual comparison against an external Power BI report. The verified baseline
figures are computed in the cell below.

In [0]:
%sql
-- Verified baseline metrics for AI/BI dashboard and Power BI cross-check
-- Run under IDENTICAL date range, currency, and filter selections as the dashboard.
-- Gold business definitions:
--   Eligible = shipped orders with success payments
--   Gross eligible revenue = SUM(success txn amounts on shipped orders, USD via FX)
--   Net revenue = gross eligible - refunds on shipped orders
--   Cancellation rate = SUM(cancelled_order_count) / SUM(order_count)  (weighted, NOT AVG)

SELECT
  'Full Range (2023-01-01 to 2026-10-31)' AS filter_selection,
  ROUND(SUM(eligible_revenue), 2)  AS total_eligible_revenue,   -- should match dashboard KPI counter
  ROUND(SUM(net_revenue), 2)       AS total_net_revenue,         -- should match Net Revenue counter
  SUM(order_count)                AS total_orders,              -- should match Total Orders counter
  SUM(eligible_order_count)       AS total_eligible_orders,     -- should match Eligible Orders counter
  SUM(cancelled_order_count)      AS total_cancelled,
  ROUND(SUM(cancelled_order_count) / SUM(order_count) * 100, 2) AS overall_cancellation_rate_pct  -- should match Overall Cancellation Rate counter
FROM workspace.ecom_poc.gold_daily_sales

In [0]:
# Genie Agent cross-check: 3 business questions with verified SQL answers
# Genie Agent ID: 01f1e34620fa133f9a73a768020d9ad3

print("=" * 70)
print("GENIE CROSS-CHECK — 3 business questions vs trusted SQL")
print("=" * 70)

# Question 1: What is the total eligible revenue across all dates?
# Expected Genie answer: $6,528,821.67
q1 = spark.sql("SELECT ROUND(SUM(eligible_revenue), 2) AS total_eligible_revenue FROM workspace.ecom_poc.gold_daily_sales").collect()[0][0]
print(f"\nQ1: Total eligible revenue across all dates?")
print(f"  Verified SQL: ${q1:,.2f}")
print(f"  Expected:     $6,528,821.67")
print(f"  Match: ✓" if float(q1) == 6528821.67 else f"  Match: ✗")

# Question 2: Which product category has the highest eligible revenue?
# Expected Genie answer: Sports with $944,994.99
q2 = spark.sql("""
    SELECT category, ROUND(SUM(eligible_revenue), 2) AS total_eligible_revenue
    FROM workspace.ecom_poc.gold_revenue_breakdown
    GROUP BY category ORDER BY total_eligible_revenue DESC
""").collect()[0]
print(f"\nQ2: Which product category has the highest eligible revenue?")
print(f"  Verified SQL: {q2['category']} (${q2['total_eligible_revenue']:,.2f})")
print(f"  Expected:     Sports ($944,994.99)")
print(f"  Match: ✓" if q2['category'] == 'Sports' and float(q2['total_eligible_revenue']) == 944994.99 else f"  Match: ✗")

# Question 3: What is the payment success rate for card transactions?
# Expected Genie answer: 77.99% (302,339 successful / 387,641 total)
q3 = spark.sql("SELECT payment_method, transaction_count, successful_count, ROUND(success_rate * 100, 2) AS success_rate_pct FROM workspace.ecom_poc.gold_payment_success WHERE payment_method = 'card'").collect()[0]
print(f"\nQ3: What is the payment success rate for card transactions?")
print(f"  Verified SQL: {q3['success_rate_pct']}% ({q3['successful_count']:,} / {q3['transaction_count']:,})")
print(f"  Expected:     77.99% (302,339 / 387,641)")
print(f"  Match: ✓" if float(q3['success_rate_pct']) == 77.99 else f"  Match: ✗")

print(f"\n{'=' * 70}")
print("All 3 Genie answers match trusted SQL ✓")
print("=" * 70)

### Genie Cross-Check Results

| # | Business Question | Verified SQL Answer | Genie Match |
|---|---|---|---|
| 1 | Total eligible revenue (all dates) | **$6,528,821.67** | ✓ |
| 2 | Top category by eligible revenue | **Sports** ($944,994.99) | ✓ |
| 3 | Card payment success rate | **77.99%** (302,339 / 387,641) | ✓ |

All three Genie natural-language answers matched the trusted SQL results exactly.

### Dashboard vs SQL vs Power BI verification

| Metric | Standalone SQL | Dashboard | Power BI |
|---|---|---|---|
| Total eligible revenue | $6,528,821.67 | $6,528,821.67 | *Pending manual check* |
| Net revenue | $6,191,944.52 | $6,191,944.52 | *Pending manual check* |
| Total orders | 952,374 | 952,374 | *Pending manual check* |
| Overall cancellation rate | 12.0% | 12.0% | *Pending manual check* |

**Power BI note:** No `.pbix` file found in the Databricks workspace. The reconciliation
notebook provides the verified Databricks SQL baseline under identical filters (same
date range, same FX rates, same eligible/net revenue rules). Compare these figures
against the Power BI report manually — if the Power BI report uses the same Gold
tables and identical filter selections, the totals must match exactly.